<a href="https://colab.research.google.com/github/ch-manasa/AI-and-ML/blob/main/RiskGuardian_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Business Context:**

In an era of rapid digital transformation, organizations across industries such as finance, healthcare, and energy face unprecedented risks that can impact their operations, reputation, and compliance. These risks range from cybersecurity breaches that compromise sensitive data to financial irregularities that disrupt business continuity. Managing these risks effectively requires not only real-time insights but also predictive capabilities to anticipate potential threats and mitigate their impact.

RiskGuardian Solutions, a global leader in consulting and risk management services, helps clients navigate these challenges by providing tailored solutions. RiskGuardian’s clients depend on advanced risk management frameworks to safeguard their operations, maintain regulatory compliance, and protect their assets. With the growing complexity of the risk landscape, accurately identifying and classifying potential risks has become a critical need for organizations aiming to remain resilient and competitive.

**Objective**

To meet the increasing demand for predictive risk management solutions, RiskGuardian has tasked its Data Science team with developing a machine learning model capable of classifying potential risks across two key domains: cybersecurity and financial.

As a Data Scientist at RiskGuardian, your goal is to analyze a dataset of historical risk events and develop a predictive model that can:

Classify potential risks as either cybersecurity or financial. Enhance the accuracy of risk identification to support proactive decision-making.

This solution will enable RiskGuardian's clients to anticipate and address emerging risks, ensuring operational resilience, regulatory compliance, and long-term business success.

**Goal:** The goal is to predict the label (0 or 1) for each risk event in the test dataset using prompt engineering with a pre-trained large language model (LLM). Your task is to correctly classify the instances where the risk event is Cybersecurity or Financial.

**Metric to Measure Performance:** The performance of your model will be evaluated using Accuracy, which is calculated as the ratio of correctly predicted labels to the total number of labels in the test dataset. The higher the accuracy, the better the model.

In [ ]:
!pip install -q -U google-generativeai
!pip install -q sentence-transformers faiss-cpu gensim lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re, time, json, hashlib, pickle, os, warnings

import google.generativeai as genai
from google.colab import userdata

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix

import lightgbm as lgb
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sentence_transformers import SentenceTransformer
import faiss

warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
DRIVE_BASE  = '/content/drive/MyDrive/AI&ML/Hacakathon'
TRAIN_PATH  = f'/content/drive/MyDrive/Datasets/combined_risk_train_(4)_(1).csv'
TEST_PATH   = f'/content/drive/MyDrive/Datasets/combined_risk_test_(5)_(1).csv'
SUBMIT_PATH = f'/content/drive/MyDrive/Datasets/submission_final.csv'
CACHE_PATH  = f'/content/drive/MyDrive/Datasets/gemini25_cache.pkl'

# ── Gemini 2.5 Flash free tier: conservative settings ──────────────────
GEMINI_MODEL      = "gemini-2.5-flash"  # Gemini 2.5 Flash
MAX_GEMINI_CALLS  = 20      # Conservative: stays within even 20 RPD accounts
                             # Increase to 50-200 if your account allows more
UNCERTAINTY_TH    = 0.12    # Call Gemini when |prob − 0.5| < 0.12 (hardest cases)
GEMINI_SLEEP_SEC  = 8    # 6.5s gap → ~9.2 RPM (limit is 10 RPM)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Config ready | Gemini calls budget: {MAX_GEMINI_CALLS}")

Config ready | Gemini calls budget: 20


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

train_df.columns = train_df.columns.str.strip().str.lower()
test_df.columns  = test_df.columns.str.strip().str.lower()

print(f"Train : {len(train_df):,} rows")
print(f"Test  : {len(test_df):,} rows")
print(f"Labels: {train_df['label'].value_counts().to_dict()}")
print(f"Avg text length: {train_df['text'].str.len().mean():.0f} chars")

Train : 14,830 rows
Test  : 70 rows
Labels: {0: 7466, 1: 7364}
Avg text length: 2004 chars


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'[^\w\s\.\-\%]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['clean'] = train_df['text'].apply(clean_text)
test_df['clean']  = test_df['text'].apply(clean_text)
print("Text cleaned")

Text cleaned


In [ ]:
# Rule based classification

REUTERS_MARKERS = ['(reuters)', 'reuters -', ' reuters)', 'told reuters',
                   'said reuters', 'according to reuters']

# Partisan blog markers → social/safety/political blog = label 0
BLOG_MARKERS    = ['featured image', '@realdonaldtrump', 'getty images',
                   'addictinginfo', 'screen capture', 'featured photo',
                   'via screenshot', 'via youtube', 'image via']


def rule_classify(text):
    """
    Returns 0, 1, or -1 (unknown → goes to ML).
    Only assign a rule label when confidence is near-certain.
    """
    t = str(text).lower()
    if any(m in t for m in REUTERS_MARKERS):
        return 1
    if any(m in t for m in BLOG_MARKERS):
        return 0
    return -1   # ML will handle this

train_df['rule_label'] = train_df['text'].apply(rule_classify)
test_df['rule_label']  = test_df['text'].apply(rule_classify)

rule_known_train = train_df[train_df['rule_label'] != -1]
rule_acc = (rule_known_train['rule_label'] == rule_known_train['label']).mean()
ml_train_mask = train_df['rule_label'] == -1
ml_test_mask  = test_df['rule_label']  == -1

print(f"\nRule-based coverage: {(~ml_train_mask).sum():,} / {len(train_df):,} "
      f"({(~ml_train_mask).mean():.1%}) at {rule_acc:.2%} accuracy")
print(f"Needs ML: {ml_train_mask.sum():,} train | {ml_test_mask.sum():,} test samples")
print(f" ML subset label dist: {train_df[ml_train_mask]['label'].value_counts().to_dict()}")


Rule-based coverage: 9,740 / 14,830 (65.7%) at 98.93% accuracy
Needs ML: 5,090 train | 20 test samples
 ML subset label dist: {0: 2593, 1: 2497}


In [ ]:
# word2vec - ML on subset data
all_texts = pd.concat([train_df['clean'], test_df['clean']], ignore_index=True)
tokenized = [simple_preprocess(t) for t in all_texts]

w2v = Word2Vec(
    sentences   = tokenized,
    vector_size = 150,
    window      = 8,        # Wide window — captures broad topic context
    min_count   = 2,
    sg          = 1,        # Skip-gram: better for rare domain words
    workers     = 4,
    epochs      = 15,
    seed        = RANDOM_SEED
)
print(f"Word2Vec vocab: {len(w2v.wv):,} words")

# Verify semantic clusters formed
print("  Checking semantic clusters:")
for word in ['malware', 'portfolio', 'vulnerability', 'liquidity']:
    if word in w2v.wv:
        nbrs = [w for w, _ in w2v.wv.most_similar(word, topn=3)]
        print(f"    '{word}' → {nbrs}")

def doc_to_vec(text, model, dim=150):
    tokens = simple_preprocess(str(text))
    vecs   = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

train_w2v = np.array([doc_to_vec(t, w2v) for t in train_df['clean']])
test_w2v  = np.array([doc_to_vec(t, w2v) for t in test_df['clean']])
print(f"W2V embeddings: train={train_w2v.shape} | test={test_w2v.shape}")

Word2Vec vocab: 55,121 words
  Checking semantic clusters:
    'malware' → ['malwar', 'mlware', 'desktop']
    'portfolio' → ['ortfolio', 'porfolio', 'portfoli']
    'vulnerability' → ['vulneability', 'vulnerabilit', 'firewalls']
    'liquidity' → ['liquidty', 'liquidit', 'iquidity']
W2V embeddings: train=(14830, 150) | test=(70, 150)


In [ ]:
#SBERT

sbert = SentenceTransformer('all-MiniLM-L6-v2')

train_emb = sbert.encode(train_df['clean'].tolist(),
                          batch_size=64, show_progress_bar=True,
                          convert_to_numpy=True)
test_emb  = sbert.encode(test_df['clean'].tolist(),
                          batch_size=64, show_progress_bar=True,
                          convert_to_numpy=True)

# Build FAISS vector DB for RAG retrieval
# Only index the ML-subset training samples (not rule-classified ones)
# — so retrieved examples are relevant to the hard cases
ml_train_indices = np.where(ml_train_mask.values)[0]
ml_train_emb     = train_emb[ml_train_indices].copy()
faiss.normalize_L2(ml_train_emb)
faiss_idx = faiss.IndexFlatIP(ml_train_emb.shape[1])
faiss_idx.add(ml_train_emb)
print(f"FAISS index: {faiss_idx.ntotal:,} vectors from ML-subset "
      f"(dim={ml_train_emb.shape[1]})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/232 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

FAISS index: 5,090 vectors from ML-subset (dim=384)


In [ ]:
# Feature engineering for ML

# Word + trigram TF-IDF: "use of force", "financial risk", "data breach"
tfidf_word = TfidfVectorizer(
    max_features = 60_000,
    ngram_range  = (1, 3),
    min_df       = 2,
    max_df       = 0.95,
    sublinear_tf = True
)
# Char TF-IDF: handles misspellings in synthetic noisy text
tfidf_char = TfidfVectorizer(
    max_features = 30_000,
    analyzer     = 'char_wb',
    ngram_range  = (3, 5),
    min_df       = 3,
    sublinear_tf = True
)

X_tr_w = tfidf_word.fit_transform(train_df['clean'])
X_te_w = tfidf_word.transform(test_df['clean'])
X_tr_c = tfidf_char.fit_transform(train_df['clean'])
X_te_c = tfidf_char.transform(test_df['clean'])

def structural_features(df):
    """
    Features derived from EDA — encode text structure signals.
    These work even when topic signals are absent.
    """
    f = pd.DataFrame()
    t = df['text']
    tl = t.str.lower()

    # Text structure
    f['text_len']       = t.str.len()
    f['word_count']     = t.str.split().str.len()
    f['avg_word_len']   = t.apply(lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0)
    f['num_count']      = t.str.count(r'\d+')
    f['pct_sign']       = t.str.count(r'\%')
    f['dollar_sign']    = t.str.count(r'\$')
    f['upper_ratio']    = t.apply(lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1))

    # Structural signals discovered in EDA
    f['is_reuters']     = tl.str.contains(r'\(reuters\)', regex=True).astype(int)
    f['has_at_sign']    = tl.str.contains(r'@real').astype(int)
    f['has_getty']      = tl.str.contains('getty').astype(int)
    f['has_featured']   = tl.str.contains('featured image').astype(int)
    f['has_quantit']    = tl.str.contains('quantit|stress test|drawdown|scenario').astype(int)
    f['has_bullet']     = t.str.count(r'\-\s').astype(int)   # Structured bullet points

    # Soft domain signals (low weight — not hard keywords)
    # These capture TOPIC without requiring exact domain words
    cyber_soft  = ['security','threat','network','attack','vulnerab','malware',
                   'breach','firewall','encrypt','intrusion','detection','hack',
                   'cyber','ransomware','phish','exploit','anomaly','surveil']
    finance_soft = ['financial','portfolio','credit','market','liquidity','debt',
                    'invest','revenue','earning','compliance','derivative',
                    'currency','exposure','drawdown','hedge','sovereign','treasury']

    f['cyber_hits']   = tl.apply(lambda x: sum(1 for w in cyber_soft if w in x))
    f['finance_hits'] = tl.apply(lambda x: sum(1 for w in finance_soft if w in x))
    f['domain_ratio'] = f['cyber_hits'] / (f['cyber_hits'] + f['finance_hits'] + 1)

    return f

scaler  = StandardScaler()
tr_feat = csr_matrix(scaler.fit_transform(structural_features(train_df)))
te_feat = csr_matrix(scaler.transform(structural_features(test_df)))

# Combine: TF-IDF + structural + W2V + SBERT
X_train = hstack([X_tr_w, X_tr_c, tr_feat,
                  csr_matrix(train_w2v),
                  csr_matrix(train_emb)])
X_test  = hstack([X_te_w, X_te_c, te_feat,
                  csr_matrix(test_w2v),
                  csr_matrix(test_emb)])
y_train = train_df['label'].values
print(f"Feature matrix: train={X_train.shape} | test={X_test.shape}")


Feature matrix: train=(14830, 90550) | test=(70, 90550)


In [ ]:
# Train ML Model

lr = LogisticRegression(
    C=3.0, max_iter=1000, solver='saga', n_jobs=-1, random_state=RANDOM_SEED
)
lgbm = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.04, num_leaves=63,
    subsample=0.8, colsample_bytree=0.7, min_child_samples=20,
    reg_alpha=0.1, reg_lambda=0.1,
    n_jobs=-1, verbose=-1, random_state=RANDOM_SEED
)

cv       = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
oof_lr   = np.zeros(len(y_train))
oof_lgbm = np.zeros(len(y_train))

print("\nCross-Validation Results:")
for name, model, oof in [("LogReg  ", lr, oof_lr), ("LightGBM", lgbm, oof_lgbm)]:
    fold_aucs = []
    for tr_idx, val_idx in cv.split(X_train, y_train):
        model.fit(X_train[tr_idx], y_train[tr_idx])
        p = model.predict_proba(X_train[val_idx])[:, 1]
        oof[val_idx] = p
        fold_aucs.append(roc_auc_score(y_train[val_idx], p))
    auc  = roc_auc_score(y_train, oof)
    rmse = np.sqrt(mean_squared_error(y_train, oof))
    acc  = accuracy_score(y_train, (oof >= 0.5).astype(int))
    print(f"  {name} | AUC={auc:.4f} | RMSE={rmse:.4f} | Acc={acc:.4f}")
    print(f"          Folds: {[f'{s:.3f}' for s in fold_aucs]}")

lr.fit(X_train, y_train)
lgbm.fit(X_train, y_train)

ensemble_oof = 0.35 * oof_lr + 0.65 * oof_lgbm
print(f"\nEnsemble OOF | AUC={roc_auc_score(y_train, ensemble_oof):.4f} | "
      f"RMSE={np.sqrt(mean_squared_error(y_train, ensemble_oof)):.4f} | "
      f"Acc={accuracy_score(y_train, (ensemble_oof>=0.5).astype(int)):.4f}")

# ML probabilities for test set
ml_proba = (0.35 * lr.predict_proba(X_test)[:, 1] +
            0.65 * lgbm.predict_proba(X_test)[:, 1])


Cross-Validation Results:
  LogReg   | AUC=1.0000 | RMSE=0.0203 | Acc=0.9995
          Folds: ['1.000', '1.000', '1.000', '1.000', '1.000']
  LightGBM | AUC=1.0000 | RMSE=0.0169 | Acc=0.9997
          Folds: ['1.000', '1.000', '1.000', '1.000', '1.000']

Ensemble OOF | AUC=1.0000 | RMSE=0.0163 | Acc=0.9997


In [ ]:
final_proba = ml_proba.copy()

# Override rule-covered samples with high-confidence rule labels
for i, row in test_df.iterrows():
    if row['rule_label'] != -1:
        # Push strongly toward the rule label (keep small ML weight for robustness)
        rule_p = float(row['rule_label'])
        final_proba[i] = 0.92 * rule_p + 0.08 * ml_proba[i]

print(f"\n Combined predictions:")
print(f"  Rule-covered test samples : {(test_df['rule_label']!=-1).sum()} / {len(test_df)}")
print(f"  ML-only test samples      : {(test_df['rule_label']==-1).sum()} / {len(test_df)}")


 Combined predictions:
  Rule-covered test samples : 50 / 70
  ML-only test samples      : 20 / 70


In [ ]:
# Save Model
import pickle, numpy as np

checkpoint = {
    # Predictions
    'ml_proba'    : ml_proba,
    'final_proba' : final_proba,
    'oof_lr'      : oof_lr,
    'oof_lgbm'    : oof_lgbm,

    # Trained models
    'lr'          : lr,
    'lgbm'        : lgbm,
    'tfidf_word'  : tfidf_word,
    'tfidf_char'  : tfidf_char,
    'scaler'      : scaler,

    # Data
    'test_df'     : test_df,
    'train_df'    : train_df,

    # Embeddings (expensive to recompute)
    'train_emb'   : train_emb,
    'test_emb'    : test_emb,
    'train_w2v'   : train_w2v,
    'test_w2v'    : test_w2v,
}

CKPT_DIR = "/content/drive/MyDrive/AI&ML/Hacakathon"
CKPT_PATH = os.path.join(CKPT_DIR, "checkpoint.pkl")

# 3. Create the folder structure if it doesn't exist
os.makedirs(CKPT_DIR, exist_ok=True)

with open(CKPT_PATH, 'wb') as f:
    pickle.dump(checkpoint, f)

print(f"  Checkpoint saved")
print(f"  ml_proba    : min={ml_proba.min():.3f} max={ml_proba.max():.3f} mean={ml_proba.mean():.3f}")
print(f"  final_proba : min={final_proba.min():.3f} max={final_proba.max():.3f} mean={final_proba.mean():.3f}")
print(f"  Label dist  : 0={( ml_proba < 0.5).sum()} | 1={(ml_proba >= 0.5).sum()}")

  Checkpoint saved
  ml_proba    : min=0.000 max=1.000 mean=0.514
  final_proba : min=0.000 max=1.000 mean=0.514
  Label dist  : 0=34 | 1=36


In [ ]:
if 'final_proba' not in dir() or final_proba is None or final_proba.max() == 0:
    print("⚠ Session died — reloading checkpoint...")
    CKPT_PATH = "/content/drive/MyDrive/AI&ML/Hacakathon/checkpoint.pkl"
    with open(CKPT_PATH, 'rb') as f:
        ckpt = pickle.load(f)

    ml_proba    = ckpt['ml_proba']
    final_proba = ckpt['final_proba']
    lr          = ckpt['lr']
    lgbm        = ckpt['lgbm']
    test_df     = ckpt['test_df']
    train_df    = ckpt['train_df']
    train_emb   = ckpt['train_emb']
    test_emb    = ckpt['test_emb']

    print(f" Restored | final_proba mean={final_proba.mean():.3f} | label dist: 0={( final_proba<0.5).sum()} 1={(final_proba>=0.5).sum()}")
else:
    print(f" Session intact | final_proba mean={final_proba.mean():.3f}")

In [ ]:
# RAG

def retrieve_examples(query_text, k=5):
    """
    Returns semantically similar examples — even for keyword-free texts.
    This is why "Arpaio pardon" will retrieve similar legal/political texts
    that are labelled 1, guiding Gemini to the right answer.
    """
    vec = sbert.encode([query_text], convert_to_numpy=True)
    faiss.normalize_L2(vec)
    scores, idxs = faiss_idx.search(vec, k)
    actual_indices = ml_train_indices[idxs[0]]
    return [
        (str(train_df.iloc[i]['text']),
         int(train_df.iloc[i]['label']),
         float(s))
        for i, s in zip(actual_indices, scores[0])
    ]

In [ ]:
def build_prompt(query_text, retrieved):
    # Extract only structural/topic signals, never raw political content
    words = str(query_text).lower().split()

    # Topic signals that are safe to send
    legal_words    = [w for w in words if w in {'court','judge','lawsuit','settlement',
                      'ruling','verdict','regulation','compliance','fine','penalty',
                      'audit','filing','contract','bankruptcy','merger','tariff'}]
    finance_words  = [w for w in words if w in {'market','stock','bond','bank','fund',
                      'trade','gdp','budget','debt','revenue','investor','portfolio'}]
    security_words = [w for w in words if w in {'hack','breach','malware','cyber',
                      'surveillance','privacy','police','arrest','threat','attack'}]

    topic_hint = f"Legal signals: {legal_words[:5]}, Finance signals: {finance_words[:5]}, Security signals: {security_words[:5]}"
    length     = len(str(query_text))
    has_quotes = str(query_text).count('"') > 3

    ex_block = ""
    for ex_text, ex_label, score in retrieved[:4]:
        ex_words = str(ex_text).lower().split()
        ex_legal = [w for w in ex_words if w in {'court','judge','lawsuit','settlement','ruling','regulation'}][:3]
        ex_fin   = [w for w in ex_words if w in {'market','bank','trade','budget','debt','investor'}][:3]
        domain   = "FINANCIAL/LEGAL (1)" if ex_label == 1 else "SECURITY/SAFETY (0)"
        ex_block += f"- {domain} | legal={ex_legal} | finance={ex_fin} | len={len(str(ex_text))}\n"

    return f"""You are a risk domain classifier. Given topic signals extracted from a news article, classify it.

FINANCIAL/LEGAL RISK (label 1): court cases, economic policy, regulatory actions, financial markets, trade
SECURITY/SAFETY RISK (label 0): cybersecurity, surveillance, law enforcement, civil rights, social unrest

Similar training examples (topic signals only):
{ex_block}
Article signals to classify:
{topic_hint}
Article length: {length} chars | Has many quotes: {has_quotes}

Reply ONLY with JSON: {{"label": <0 or 1>, "confidence": <0.5 to 1.0>}}"""


In [ ]:
# Gemini API Key

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
gemini = genai.GenerativeModel(GEMINI_MODEL)

CACHE = {}
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'rb') as f:
        CACHE = pickle.load(f)
    print(f" Loaded {len(CACHE):,} cached responses (cost = 0 tokens)")

import re as _re

def gemini_classify(text, retrieved, retries=4):
    cache_key = hashlib.md5(text[:200].encode()).hexdigest()
    if cache_key in CACHE:
        return {**CACHE[cache_key], 'cached': True}

    prompt = build_prompt(text, retrieved)
    for attempt in range(retries):
        try:
            resp = gemini.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    temperature=0.0, max_output_tokens=50
                ),
                safety_settings={
                    "HARM_CATEGORY_HARASSMENT":        "BLOCK_NONE",
                    "HARM_CATEGORY_HATE_SPEECH":       "BLOCK_NONE",
                    "HARM_CATEGORY_SEXUALLY_EXPLICIT": "BLOCK_NONE",
                    "HARM_CATEGORY_DANGEROUS_CONTENT": "BLOCK_NONE",
                }
            )

            candidate = resp.candidates[0]
            if candidate.finish_reason not in (1,):
                print(f"   Skipped (finish_reason={candidate.finish_reason})")
                return {"label": -1, "confidence": 0.0}

            raw    = re.sub(r'```json|```', '', resp.text.strip()).strip()
            result = json.loads(raw)
            result['label'] = int(result['label'])
            CACHE[cache_key] = result
            return result

        except json.JSONDecodeError:
            label  = 1 if '"label":1' in raw.replace(' ', '') else 0
            result = {"label": label, "confidence": 0.6}
            CACHE[cache_key] = result
            return result

        except Exception as e:
            err_str = str(e)
            if '429' in err_str or 'quota' in err_str.lower():
                # Parse retry-after from error message if present
                match = _re.search(r'retry in (\d+)', err_str)
                wait  = int(match.group(1)) + 5 if match else (2 ** attempt) * 15
                print(f"   Rate limit (attempt {attempt+1}) — waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"   Error: {e}")
                return {"label": -1, "confidence": 0.0}

    return {"label": -1, "confidence": 0.0}

In [ ]:
# Only uncertain ML-only samples
ml_only_idx   = np.where(test_df['rule_label'].values == -1)[0]
uncertainty   = np.abs(final_proba - 0.5)
uncertain_ml  = ml_only_idx[np.argsort(uncertainty[ml_only_idx])]  # hardest first
call_idx      = uncertain_ml[:MAX_GEMINI_CALLS]

print(f"   ML-only test samples   : {len(ml_only_idx)}")
print(f"   Of which uncertain     : {(uncertainty[ml_only_idx] < UNCERTAINTY_TH).sum()}")
print(f"   Calling Gemini on      : {len(call_idx)} (hardest first)")
print(f"   Est. tokens            : ~{len(call_idx) * 600:,}")

llm_labels = {}
llm_confs  = {}
api_calls  = 0

for i, idx in enumerate(call_idx):
    query_text = test_df.iloc[idx]['text']
    retrieved  = retrieve_examples(query_text, k=5)
    result     = gemini_classify(query_text, retrieved)

    if result['label'] != -1:
        llm_labels[idx] = result['label']
        llm_confs[idx]  = result.get('confidence', 0.7)
        if not result.get('cached', False):
            api_calls += 1
            time.sleep(GEMINI_SLEEP_SEC)

    if (i+1) % 5 == 0 or (i+1) == len(call_idx):
        print(f"  → {i+1}/{len(call_idx)} | API calls: {api_calls} | Cached: {len(CACHE)}")

print(f"\n Gemini labelled: {len(llm_labels)} | Real API calls: {api_calls}")


   ML-only test samples   : 20
   Of which uncertain     : 0
   Calling Gemini on      : 20 (hardest first)
   Est. tokens            : ~12,000
   Skipped (finish_reason=2)


   Rate limit (attempt 1) — waiting 44s...


   Rate limit (attempt 2) — waiting 60s...


KeyboardInterrupt: 

In [ ]:
for idx in call_idx:
    if idx not in llm_labels:
        llm_labels[idx] = int(final_proba[idx] >= 0.5)
        llm_confs[idx]  = float(abs(final_proba[idx] - 0.5) + 0.5)

print(f"\n Total labelled: {len(llm_labels)} | Gemini: {api_calls} | ML fallback: {len(call_idx)-api_calls}")


 Total labelled: 20 | Gemini: 0 | ML fallback: 20


In [ ]:
final_preds = (final_proba >= 0.5).astype(int)

submission = pd.DataFrame({
    'id':    test_df['id'].astype(int),
    'label': final_preds
})
submission.to_csv(SUBMIT_PATH, index=False)

print(f"Shape: {submission.shape}")
print(f"Label 0: {(final_preds==0).sum()} | Label 1: {(final_preds==1).sum()}")
print(submission.head(10).to_string(index=False))


Shape: (70, 2)
Label 0: 34 | Label 1: 36
 id  label
  0      0
  1      0
  2      0
  3      1
  4      1
  5      1
  6      1
  7      1
  8      1
  9      0


In [ ]:
# Save cache
with open(CACHE_PATH, 'wb') as f:
    pickle.dump(CACHE, f)
print(f"\n Cache saved ({len(CACHE)} entries) — next run costs 0 API calls!")

print(f"""
 -------------------------------------------------------------
  Step 1  Rule-based  │ 65% coverage │ ~99% accuracy
  Step 2  ML ensemble │ 35% coverage │ see CV results above
  Step 3  Gemini RAG  │ uncertain ML │ {api_calls} API calls
---------------------------------------------------------------
  Gemini API calls : {api_calls:<42}
  Tokens used est. : {api_calls*600:<42,}
  Cached (free)    : {len(llm_labels)-api_calls:<42}
---------------------------------------------------------------
""")


 Cache saved (0 entries) — next run costs 0 API calls!

 -------------------------------------------------------------
  Step 1  Rule-based  │ 65% coverage │ ~99% accuracy          
  Step 2  ML ensemble │ 35% coverage │ see CV results above   
  Step 3  Gemini RAG  │ uncertain ML │ 0 API calls  
---------------------------------------------------------------
  Gemini API calls : 0                                                                   
  Tokens used est. : 0                                               
  Cached (free)    : 20                                        
--------------------------------------------------------------- 



In [ ]:
# Run this after final_preds is computed

from sklearn.metrics import (accuracy_score, roc_auc_score,
                              mean_squared_error, confusion_matrix,
                              classification_report)

# ── OOF (Out-of-Fold) estimates on TRAIN — best proxy for test performance ──
ensemble_oof  = 0.35 * oof_lr + 0.65 * oof_lgbm
oof_preds     = (ensemble_oof >= 0.5).astype(int)

oof_acc       = accuracy_score(y_train, oof_preds)
oof_rmse      = np.sqrt(mean_squared_error(y_train, oof_preds))
oof_auc       = roc_auc_score(y_train, ensemble_oof)

print("=" * 50)
print("  OOF PERFORMANCE (proxy for test score)")
print("=" * 50)
print(f"  Accuracy : {oof_acc:.4f}  ({oof_acc*100:.2f}%)")
print(f"  RMSE     : {oof_rmse:.4f}  ← this is leaderboard estimate")
print(f"  AUC      : {oof_auc:.4f}")
print()

# ── What leaderboard position this maps to ──
best_rmse = 0.1195
print(f"  Leaderboard best RMSE : {best_rmse}")
print(f"  Your estimated RMSE   : {oof_rmse:.4f}")
print(f"  Gap                   : {oof_rmse - best_rmse:+.4f}")
print()

# ── Confusion matrix ──
cm = confusion_matrix(y_train, oof_preds)
print("  Confusion Matrix (OOF):")
print(f"              Pred 0    Pred 1")
print(f"  Actual 0  :  {cm[0,0]:5d}     {cm[0,1]:5d}")
print(f"  Actual 1  :  {cm[1,0]:5d}     {cm[1,1]:5d}")
print()

# ── Submission distribution check ──
print("=" * 50)
print("  SUBMISSION FILE CHECK")
print("=" * 50)
print(f"  Total rows : {len(final_preds)}")
print(f"  Label 0    : {(final_preds==0).sum()} ({(final_preds==0).mean():.1%})")
print(f"  Label 1    : {(final_preds==1).sum()} ({(final_preds==1).mean():.1%})")
print()

# ── Train label distribution for comparison ──
print(f"  Train Label 0 : {(y_train==0).sum()} ({(y_train==0).mean():.1%})")
print(f"  Train Label 1 : {(y_train==1).sum()} ({(y_train==1).mean():.1%})")
print()

# ── Red flag check ──
if (final_preds==0).all() or (final_preds==1).all():
    print("  WARNING: All predictions are the same label — something is wrong!")
else:
    print("  Predictions look healthy (mix of 0s and 1s)")

if abs((final_preds==1).mean() - (y_train==1).mean()) > 0.2:
    print("  WARNING: Test label distribution very different from train")
else:
    print("  Test distribution roughly matches train distribution")

  OOF PERFORMANCE (proxy for test score)
  Accuracy : 0.9997  (99.97%)
  RMSE     : 0.0184  ← this is your leaderboard estimate
  AUC      : 1.0000

  Leaderboard best RMSE : 0.1195
  Your estimated RMSE   : 0.0184
  Gap                   : -0.1011

  Confusion Matrix (OOF):
              Pred 0    Pred 1
  Actual 0  :   7466         0
  Actual 1  :      5      7359

  SUBMISSION FILE CHECK
  Total rows : 70
  Label 0    : 34 (48.6%)
  Label 1    : 36 (51.4%)

  Train Label 0 : 7466 (50.3%)
  Train Label 1 : 7364 (49.7%)

  Predictions look healthy (mix of 0s and 1s)
  Test distribution roughly matches train distribution
